In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

tqdm.pandas()

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)


np.random.seed(42)  
num_samples = 200
lats = np.random.uniform(-90, 90, num_samples) 
lons = np.random.uniform(-180, 180, num_samples)  
start_date = datetime(2000, 1, 1)
end_date = datetime(2006, 12, 31)
dates = [random_date(start_date, end_date) for _ in range(num_samples)]

df = pd.DataFrame({
    'Latitude': lats,
    'Longitude': lons,
    'Sample Date': dates
})

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'])

    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2000-01-01/2006-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
   
    items = search.item_collection()
    if not items:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })
    try:
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
       
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
       
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
   
    except Exception as e:
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })

landsat_features = df.progress_apply(compute_Landsat_values, axis=1)
df_with_features = pd.concat([df[['Latitude', 'Longitude', 'Sample Date']], landsat_features], axis=1)

eps = 1e-10
df_with_features['NDMI'] = (df_with_features['nir'] - df_with_features['swir16']) / (df_with_features['nir'] + df_with_features['swir16'] + eps)
df_with_features['MNDWI'] = (df_with_features['green'] - df_with_features['swir16']) / (df_with_features['green'] + df_with_features['swir16'] + eps)

# Save to CSV
df_with_features.to_csv('landsat_features_2000_2006_200_samples.csv', index=False)

100%|██████████| 200/200 [13:29<00:00,  4.05s/it]


In [2]:
import pandas as pd

df = pd.read_csv('landsat_features_2000_2006_200_samples.csv')
pd.set_option('display.max_rows', None)
print(df.head())

    Latitude   Longitude Sample Date     nir   green  swir16  swir22  \
0 -22.582779   51.131393  2002-05-22  7962.0  8326.0  7835.0  7785.0   
1  81.128575 -149.709613  2004-12-26     NaN     NaN     NaN     NaN   
2  41.758910 -121.813663  2003-07-20     NaN     NaN     NaN     NaN   
3  17.758527  143.479508  2005-10-10  7555.0  7907.0  7448.0  7367.0   
4 -61.916645   38.314461  2001-08-18     NaN     NaN     NaN     NaN   

       NDMI     MNDWI  
0  0.008040  0.030382  
1       NaN       NaN  
2       NaN       NaN  
3  0.007132  0.029893  
4       NaN       NaN  
